In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [2]:
print("Script started for 3-class sentiment analysis...")


print("Checking for NLTK data packages...")
nltk_packages = ['stopwords', 'wordnet', 'omw-1.4']
for package in nltk_packages:
    try:
        nltk.data.find(f'corpora/{package}.zip')
    except LookupError:
        print(f"Downloading NLTK package: {package}")
        nltk.download(package)
print("NLTK packages are up to date.")

Script started for 3-class sentiment analysis...
Checking for NLTK data packages...
NLTK packages are up to date.


In [3]:
try:
    df = pd.read_csv('amazon_reviews.csv')
    print("Data loaded successfully.")
except FileNotFoundError:
    print("\n[ERROR] 'amazon_reviews.csv' not found.")
    print("Please download a suitable dataset and place it in the project directory.")
    exit()

df = df[['reviews.text', 'reviews.rating']].dropna()

Data loaded successfully.


In [4]:
def map_sentiment(rating):
    if rating <= 2:
        return 0  # Negative
    elif rating == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df['sentiment'] = df['reviews.rating'].apply(map_sentiment)

In [5]:
min_count = df['sentiment'].value_counts().min()
df_negative = df[df['sentiment'] == 0].sample(min_count, random_state=42)
df_neutral = df[df['sentiment'] == 1].sample(min_count, random_state=42)
df_positive = df[df['sentiment'] == 2].sample(min_count, random_state=42)
df_balanced = pd.concat([df_negative, df_neutral, df_positive])

print(f"Dataset balanced with {min_count} reviews for each of the 3 classes.")

X = df_balanced['reviews.text']
y = df_balanced['sentiment']

Dataset balanced with 76 reviews for each of the 3 classes.


In [6]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = re.sub('[^a-zA-Z]', ' ', str(text))
    text = text.lower()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

print("Preprocessing text data...")
X_processed = X.apply(preprocess_text)

Preprocessing text data...


In [7]:

vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.7)
X_tfidf = vectorizer.fit_transform(X_processed).toarray()
print("TF-IDF vectorization complete.")

TF-IDF vectorization complete.


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
print("Training Logistic Regression model...")
model.fit(X_train, y_train)
print("Model training complete.")

Training Logistic Regression model...
Model training complete.


In [9]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report (0=Negative, 1=Neutral, 2=Positive):")
print(classification_report(y_test, y_pred))


--- Model Evaluation ---
Accuracy: 0.6957
Classification Report (0=Negative, 1=Neutral, 2=Positive):
              precision    recall  f1-score   support

           0       0.53      0.73      0.62        11
           1       0.79      0.58      0.67        19
           2       0.76      0.81      0.79        16

    accuracy                           0.70        46
   macro avg       0.69      0.71      0.69        46
weighted avg       0.72      0.70      0.70        46



In [10]:
import pandas as pd

df = pd.read_csv("amazon_reviews.csv")
df.head(100).to_csv("amazon_reviews_sample.csv", index=False)

print("Sample CSV created successfully!")

Sample CSV created successfully!
